In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Configure ADLS access
storage_key = dbutils.secrets.get(scope="azure-storage", key="storage-account-key")
spark.conf.set(
    "fs.azure.account.key.avddevstorageacc2026.dfs.core.windows.net",
    storage_key
)

ADLS_ROOT = "abfss://datalakecontainer@avddevstorageacc2026.dfs.core.windows.net"

# Define a helper function to move any dataset into Bronze
def move_to_bronze(file_name, table_name):
    input_path = f"{ADLS_ROOT}/landing/{file_name}"
    bronze_path = f"{ADLS_ROOT}/bronze/{table_name}"

     # Read parquet file
    df = spark.read.parquet(input_path)

    # Add audit columns
    df_bronze = df \
        .withColumn("ingested_at", F.current_timestamp()) \
        .withColumn("source_system", F.lit("azure_sql"))

    # Write to Bronze Delta
    df_bronze.write \
        .format("delta") \
        .mode("overwrite") \
        .save(bronze_path)

    print(f"{table_name} → Bronze layer at {bronze_path}")

# Move each dataset into Bronze
move_to_bronze("raw_sales.parquet", "raw_sales")
move_to_bronze("dim_stores.parquet", "dim_stores")
move_to_bronze("dim_products.parquet", "dim_products")

In [0]:
spark.read.format("delta").load("abfss://datalakecontainer@avddevstorageacc2026.dfs.core.windows.net/bronze/dim_stores").display()